In [ ]:
#importing torch for GPU
!pip install fpdf
import os
from fpdf import FPDF
from IPython.display import Image
import torch

print(torch.cuda.is_available())

In [ ]:
#Cloning Yolov6 rep
!git clone https://github.com/meituan/YOLOv6
%cd YOLOv6
!pip install -r requirements.txt

In [ ]:
#importing images of covid-19 dataset from github
!curl -L "https://github.com/SURYA282003/covidDetection/raw/main/covid19detection.zip"> covid19detection.zip
!unzip covid19detection.zip
!rm covid19detection.zip

In [ ]:
# Download YOLOv6 pre-trained model
%cd /content/YOLOv6
!wget https://github.com/meituan/YOLOv6/releases/download/0.4.0/yolov6s.pt

In [ ]:
!ls /content/YOLOv6/images/train

In [ ]:
#Training
%cd YOLOv6
!python tools/train.py --batch 16 --conf configs/yolov6s_finetune.py --data /content/YOLOv6/data.yaml --device 0 --epochs 40

In [ ]:
!python tools/eval.py --data /content/YOLOv6/data.yaml  --weights runs/train/exp/weights/best_ckpt.pt --device 0

In [ ]:
from google.colab import files
import zipfile
import os

# Step 1: Upload the zip file
uploaded = files.upload()

# Step 2: Extract the zip file
for file_name in uploaded.keys():
    print(f'Uploaded file: {file_name}')

    # Full path of the uploaded zip file
    zip_path = os.path.join("/content/YOLOv6", file_name)

    # Extract the zip file
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Extract all the contents into a folder
        extracted_folder = os.path.join("/content/YOLOv6", file_name.split(".")[0])
        zip_ref.extractall(extracted_folder)

    # Step 3: Print the path of the extracted folder
    print(f'Files extracted to: {extracted_folder}')

    # List the contents of the extracted folder
    extracted_files = os.listdir(extracted_folder)
    print(f'Extracted files: {extracted_files}')


In [ ]:
import os
import re  # For pattern matching


weights = "runs/train/exp/weights/best_ckpt.pt"
yaml_file = "/content/YOLOv6/data.yaml"
output_folder = "/content/YOLOv6/runs/inference/exp"
image_folder = extracted_folder


image_files = [os.path.join(image_folder, img) for img in os.listdir(image_folder) if img.endswith('.jpg')]


results = []

# Loop through the images and perform inference
for img in image_files:
    # Perform inference on the image
    # --save-txt flag makes YOLOv6 save the detection results in text files
    !python tools/infer.py --weights {weights} --source {img} --device 0 --yaml {yaml_file} --save-txt

    # Extracting the image name from the path
    img_name = os.path.basename(img)

    # YOLOv6 typically outputs the results in a folder (adjust the output path as per your setup)
    txt_output_path = os.path.join(output_folder, 'labels', f'{img_name.split(".")[0]}.txt')  # YOLOv6 saves results as .txt files

    # Initialize prediction variable
    prediction = "No COVID"

    # Check if YOLOv6 detection results exist for this image
    if os.path.exists(txt_output_path):
        # Open the YOLOv6 results file and look for the class names
        with open(txt_output_path, 'r') as result_file:
            for line in result_file:
                # Parse each line (YOLOv6 format: class_id, x_center, y_center, width, height, confidence)
                class_id = int(line.split()[0])  # Assuming first value in each line is class ID

                # Assuming that class_id corresponds to "COVID" in your model (adjust based on your model's class mapping)
                if class_id == 0:  # Assuming class_id == 1 corresponds to COVID
                    prediction = "COVID Detected"
                    break

    # Append the image path and prediction to the results list
    results.append((img, prediction))


In [ ]:
import os

# Specify the folder path
folder_path = "/content/YOLOv6/runs/inference/exp/labels"

# Get the list of all files and directories in the folder
files_in_folder = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

# Count the number of files
file_count = len(files_in_folder)

print(f"Total files in the folder: {file_count}")

In [ ]:
import os

# Specify the folder path
folder_path = "/content/YOLOv6/runs/inference/exp"

# Get the list of all files and directories in the folder
files_in_folder = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

# Count the number of files
file_count = len(files_in_folder)

print(f"Total files in the folder: {file_count}")

In [ ]:
import os
from fpdf import FPDF

# Paths to image and label folders
image_folder = "/content/YOLOv6/runs/inference/exp"
labels_folder = "/content/YOLOv6/runs/inference/exp/labels"

# Initialize PDF
pdf = FPDF()
pdf.add_page()

# Set font for the PDF
pdf.set_font("Arial", size=12)

# Add a title
pdf.cell(200, 10, txt="Image COVID Report", ln=True, align="C")

# Space between title and first content
pdf.ln(10)

# Set up column widths
image_width = 100  # Adjust as per your preference
label_width = 50  # Adjust as per your preference

# Add table header
pdf.set_font("Arial", "B", 10)
pdf.cell(image_width, 15, "Image Path", border=1, align="C")
pdf.cell(label_width, 15, "COVID Status", border=1, align="C")
pdf.ln()

# Get the list of image files and label files
image_files = sorted(os.listdir(image_folder))
label_files = sorted(os.listdir(labels_folder))



# Iterate through the image and label files
for image_file, label_file in zip(image_files, label_files):
    # Construct the full paths
    image_path = os.path.join(image_folder, image_file)
    label_path = os.path.join(labels_folder, label_file)

    # Read the label file to get the first digit
    with open(label_path, 'r') as label_file:
        label_data = label_file.readline().strip()

    # Extract the first digit from the label file to determine if COVID or not
    label_parts = label_data.split()
    is_covid = "COVID" if int(label_parts[0]) == 0 else "Not COVID"

    # Set font for the content rows
    pdf.set_font("Arial", size=10)

    # Add the image path (link) and label to the PDF
    pdf.cell(image_width, 15, txt=image_file, border=1, align="L")
    pdf.cell(label_width, 15, txt=is_covid, border=1, align="C")
    pdf.ln()

# Save the PDF
output_pdf = "/content/YOLOv6/runs/inference/exp/detection_results.pdf"
pdf.output(output_pdf)

print(f"PDF report generated: {output_pdf}")
